# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [1]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.6 MB/s eta 0:00:00


In [2]:
!pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.8 MB/s eta 0:00:00a 0:00:01


In [3]:
# -*- coding: utf-8 -*-
"""starter_notebook (1).ipynb — Pixels to Predictions DL Vision Challenge

REVISED pipeline — key changes from baseline (starter_notebook_baseline.py):
  A. Training loss computed only on assistant answer tokens (prompt masked with -100)
  B. build_prompt() never appends the answer; answer is a separate assistant turn
  C. Primary inference uses log-likelihood ranking (zero invalid outputs)
  D. Generate-based inference kept as optional fallback
  G. DEBUG mode for quick iteration
  H. Processor image-size introspection printed

Baseline Model : HuggingFaceTB/SmolVLM-500M-Instruct (~500M params)
Fine-Tuning    : LoRA / DoRA via PEFT (< 5M trainable parameters)
Scoring        : Multiple-choice log-likelihood ranking
"""

# ── 0. Install libraries (uncomment on Colab/Kaggle) ─────────────────────────
# !pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import gc
import json
import math
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
# ── Robust HF Hub download settings ──────────────────────────────────────────
# Per-request timeout (seconds) for downloads from huggingface.co. The default
# is short and silently hangs on a stalled CDN chunk (we hit this at 66% on
# Kaggle). Raise it so a slow chunk doesn't kill the whole load.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
# Use the high-throughput downloader if available (Kaggle ships hf_transfer).
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

'1'

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')



In [5]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# =============================================================================
#  CONFIG FLAGS — adjust these for experiments
# =============================================================================
MODEL_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"

IMAGE_SIZE = 336          # 224 | 336 | 384 | 448  (336 keeps training time manageable)
MAX_TEXT_TOKENS = 512     # raised from 256 — keeps most lecture content uncut
USE_METADATA = True       # include subject / grade / topic in prompt
USE_HINT = True           # include hint field
USE_LECTURE = True        # include lecture field
USE_SELF_CAPTIONS = False # self-generated captions (optional, slow)
USE_DORA = True           # DoRA: small gain, ~no inference cost
USE_GRADIENT_CHECKPOINTING = torch.cuda.is_available()

# Choice permutation: shuffle choice order each epoch and remap the answer.
# This is the single biggest fix for the position-bias overfit seen in run 1.
USE_CHOICE_PERMUTATION = True

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.10        # raised from 0.05 — extra regularization
TARGET_MODULES_MODE = "attention_mlp"
# Options: "qv_only", "attention_only", "attention_mlp", "qv_mlp"

MAX_TRAINABLE_PARAMS = 5_000_000

# Training hyperparameters
NUM_EPOCHS = 3             # raised from 2 — lower LR needs more epochs
BATCH_SIZE = 2             # keep at 2 for memory headroom with longer text
GRAD_ACCUM_STEPS = 4       # effective batch = 8
LEARNING_RATE = 1e-4       # halved from 2e-4 — fixes overfit at epoch 2
WEIGHT_DECAY = 0.05        # raised from 0.01
WARMUP_RATIO = 0.05
EVAL_EVERY_N_STEPS = 0  # validate every N optimizer steps

# Inference mode: "loglikelihood" (recommended) or "generate" (fallback)
INFERENCE_MODE = "loglikelihood"

# ── Debug mode ───────────────────────────────────────────────────────────────
DEBUG = False               # set True for fast iteration; False for full run
TRAIN_SUBSET = None          # used when DEBUG=True
VAL_SUBSET = 16            # used when DEBUG=True
TEST_SUBSET = 16            # used when DEBUG=True (None = full test set)

# Override settings in debug mode
if DEBUG:
    NUM_EPOCHS = 1
    EVAL_EVERY_N_STEPS = 0  # skip mid-epoch eval
    GRAD_ACCUM_STEPS = 1    # no accumulation in debug
    BATCH_SIZE = 1
    print(f"[DEBUG MODE] train={TRAIN_SUBSET}, val={VAL_SUBSET}, "
          f"test={'full' if TEST_SUBSET is None else TEST_SUBSET}, epochs={NUM_EPOCHS}")


In [6]:
#!cp -r /content/drive/MyDrive/pixels-to-predictions /content/

In [7]:
# ── Paths ────────────────────────────────────────────────────────────────────
# DATA_DIR = Path("/content/pixels-to-predictions")

# print(f"DATA_DIR resolved to: {DATA_DIR.resolve()}")

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/kaggle/input/competitions/pixels-to-predictions")
print(f"DATA_DIR resolved to: {DATA_DIR.resolve()}")

DATA_DIR resolved to: /kaggle/input/competitions/pixels-to-predictions


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [9]:

# =============================================================================
#  2. Load and Preprocess Data
# =============================================================================

# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# Apply debug subsetting
if DEBUG:
    if TRAIN_SUBSET:
        train_df = train_df.head(TRAIN_SUBSET).copy()
    if VAL_SUBSET:
        val_df = val_df.head(VAL_SUBSET).copy()
    if TEST_SUBSET:
        test_df = test_df.head(TEST_SUBSET).copy()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Sample submission rows: {len(sample_sub):,}")
print(f"Columns — train: {list(train_df.columns)}")
print(f"Columns — test:  {list(test_df.columns)}")

Train: 3,109 | Val: 1,048 | Test: 1,008
Sample submission rows: 1,008
Columns — train: ['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']
Columns — test:  ['id', 'image_path', 'question', 'choices', 'num_choices', 'hint', 'lecture', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


In [10]:
def resolve_image_path(row, split, data_dir=DATA_DIR):
    """Try multiple path patterns and return the first that exists."""
    image_path = row.get("image_path", "")
    if pd.isna(image_path):
        image_path = ""
    image_path = str(image_path).strip()
    filename = Path(image_path).name if image_path else ""

    candidates = []
    if image_path:
        candidates.append(data_dir / image_path)
        candidates.append(data_dir / "images" / "images" / split / filename)
        candidates.append(data_dir / "images" / "images" / image_path)
    if filename:
        candidates.append(data_dir / "images" / split / filename)
        candidates.append(data_dir / "data" / "images" / split / filename)
    row_id = row.get("id", "")
    if row_id:
        candidates.append(data_dir / "images" / "images" / split / f"{row_id}.png")
        candidates.append(data_dir / "images" / split / f"{row_id}.png")

    for p in candidates:
        if p.exists():
            return str(p)
    return str(data_dir / image_path) if image_path else ""


_test_row = train_df.iloc[0]
_resolved = resolve_image_path(_test_row, "train")
print(f"Image path check: {_test_row['image_path']} -> {_resolved} "
      f"(exists: {Path(_resolved).exists()})")

# ── 2c. Prompt Builder ───────────────────────────────────────────────────────

def _safe_str(val):
    """Return stripped string or empty string if NaN/None."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return ""
    s = str(val).strip()
    return "" if s.lower() == "nan" else s


def build_prompt(row, choices_override=None):
    """
    Build user-side prompt text (never includes the answer).
    The answer is always a separate assistant message during training.

    If ``choices_override`` is supplied, it is used in place of row["choices"].
    This lets the training dataset render permuted choices.
    """
    parts = []
    parts.append("You are solving a science multiple-choice question "
                 "using the image and text.")
    parts.append("")

    if USE_METADATA:
        subject = _safe_str(row.get("subject", ""))
        grade   = _safe_str(row.get("grade", ""))
        topic   = _safe_str(row.get("topic", ""))
        if subject:
            parts.append(f"Subject: {subject}")
        if grade:
            parts.append(f"Grade: {grade}")
        if topic:
            parts.append(f"Topic: {topic}")
        if subject or grade or topic:
            parts.append("")

    if USE_HINT:
        hint = _safe_str(row.get("hint", ""))
        if hint:
            parts.append(f"Hint:\n{hint}")
            parts.append("")

    if USE_LECTURE:
        lecture = _safe_str(row.get("lecture", ""))
        if lecture:
            parts.append(f"Lecture:\n{lecture}")
            parts.append("")

    question = _safe_str(row.get("question", ""))
    parts.append(f"Question:\n{question}")
    parts.append("")

    choices = choices_override if choices_override is not None else row["choices"]
    choices_lines = [f"{i}. {c}" for i, c in enumerate(choices)]
    parts.append("Choices:\n" + "\n".join(choices_lines))
    parts.append("")
    parts.append("Answer with only the 0-indexed integer of the correct choice.")

    return "\n".join(parts)


# Display an example prompt (user side only — no answer appended)
print("=" * 60)
print("EXAMPLE PROMPT (user message — no answer):")
print("=" * 60)
print(build_prompt(train_df.iloc[0]))
print("=" * 60)

# ── 2d. PyTorch Dataset ──────────────────────────────────────────────────────

class ScienceQADataset(Dataset):
    def __init__(self, df, split, data_dir=DATA_DIR, img_size=IMAGE_SIZE,
                 permute_choices=False):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.data_dir = data_dir
        self.img_size = img_size
        # Permute the order of multiple-choice options each time __getitem__
        # is called and remap the answer accordingly. This breaks the
        # "answer-by-position" shortcut that caused the run-1 overfit.
        self.permute_choices = permute_choices

    def __len__(self):
        return len(self.df)

    def _load_image(self, row):
        path = resolve_image_path(row, self.split, self.data_dir)
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
            return img
        except Exception as e:
            print(f"[WARN] Could not load image {path}: {e}")
            return Image.new("RGB", (self.img_size, self.img_size), (128, 128, 128))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row)

        choices = list(row["choices"])
        orig_answer = int(row["answer"]) if (
            "answer" in row and pd.notna(row.get("answer"))) else -1

        if self.permute_choices and orig_answer >= 0 and len(choices) > 1:
            order = list(range(len(choices)))
            random.shuffle(order)
            permuted_choices = [choices[i] for i in order]
            new_answer = order.index(orig_answer)
            prompt = build_prompt(row, choices_override=permuted_choices)
            answer_to_emit = new_answer
            choices_to_emit = permuted_choices
        else:
            prompt = build_prompt(row)
            answer_to_emit = orig_answer
            choices_to_emit = choices

        item = {
            "image": img,
            "prompt": prompt,        # user-side text only
            "id": row["id"],
            "num_choices": int(row["num_choices"]),
            "choices": choices_to_emit,
            "answer": answer_to_emit,
        }
        return item


# Permute choices ONLY in the training set. Val/test must stay deterministic
# so accuracy comparisons are meaningful.
train_ds = ScienceQADataset(train_df, "train",
                            permute_choices=USE_CHOICE_PERMUTATION)
val_ds   = ScienceQADataset(val_df,   "val")
test_ds  = ScienceQADataset(test_df,  "test")

print(f"\nDatasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

_sample = train_ds[0]
print(f"Sample keys: {list(_sample.keys())}")
print(f"Sample image size: {_sample['image'].size}")
print(f"Sample answer: {_sample['answer']}")

Image path check: images/train/train_07667.png -> /kaggle/input/competitions/pixels-to-predictions/images/images/train/train_07667.png (exists: True)
EXAMPLE PROMPT (user message — no answer):
You are solving a science multiple-choice question using the image and text.

Subject: natural science
Grade: grade8
Topic: literacy-in-science

Hint:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, the female frog lays eggs on a plant. When tadpoles hatch from the eggs, the male frog lets the tadpoles climb onto his back. The male then searches for water trapped in the spaces where plants' leaves meet their stems. He puts his tadpoles in these small pools of water.
If the male frog puts a tadpole into a pool with a larger tadpole, the smaller tadpole is often eaten.

In [11]:
# =============================================================================
#  3. Model Loading
# =============================================================================
from transformers import AutoProcessor, AutoModelForVision2Seq

def _load_with_retry(load_fn, *, what, attempts=4, base_wait=10):
    """Call load_fn() with retry + exponential backoff. Hardens
    AutoProcessor / AutoModelForVision2Seq against HF Hub stalls. Each retry
    benefits from huggingface_hub's partial-file cache so we resume rather
    than redownload from zero."""
    for attempt in range(1, attempts + 1):
        try:
            return load_fn()
        except Exception as e:
            if attempt == attempts:
                raise
            wait = base_wait * (2 ** (attempt - 1))
            print(f"[{what}] attempt {attempt}/{attempts} failed: "
                  f"{type(e).__name__}: {e}. Retrying in {wait}s...")
            time.sleep(wait)


print(f"\nLoading model: {MODEL_NAME}")
processor = _load_with_retry(
    lambda: AutoProcessor.from_pretrained(MODEL_NAME, resume_download=True),
    what="AutoProcessor",
)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# ── Fix H: Print processor image size config ─────────────────────────────────
print("\n--- Processor image configuration ---")
if hasattr(processor, 'image_processor'):
    ip = processor.image_processor
    for attr in ['size', 'crop_size', 'image_size', 'do_resize', 'resample']:
        if hasattr(ip, attr):
            print(f"  image_processor.{attr} = {getattr(ip, attr)}")
    print(f"  NOTE: You are resizing to {IMAGE_SIZE}x{IMAGE_SIZE} before the processor.")
    print(f"        The processor may resize again internally. Check 'size' above.")
else:
    print("  (no image_processor attribute found)")

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = _load_with_retry(
    lambda: AutoModelForVision2Seq.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        resume_download=True,
    ),
    what="AutoModelForVision2Seq",
)
if not torch.cuda.is_available():
    model.to(device)

print(f"Model loaded. Total params: {sum(p.numel() for p in model.parameters()):,}")


def truncate_prompt_text(prompt_text, max_tokens=MAX_TEXT_TOKENS):
    """
    Smart truncation that NEVER drops the question, choices, or trailing
    instruction. Long lectures and hints are truncated from their tails
    instead of the question being chopped off.

    Layout assumed (built by ``build_prompt``):
        <intro>
        <metadata block?>
        <Hint:\n...\n>
        <Lecture:\n...\n>
        Question:\n...\n
        Choices:\n...\n
        Answer with only...

    Strategy:
        1. If the whole prompt fits, return as-is.
        2. Otherwise locate the "Question:" marker and split prompt into
           pre-question (intro+metadata+hint+lecture) and post-question
           (question+choices+suffix). Post-question is sacred.
        3. Trim pre-question by the overflow amount (token-aware), keeping
           the start of pre-question (intro/metadata) and dropping from
           the end (lecture body) — that's where the bulk lives.
    """
    tokenizer = processor.tokenizer
    ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return prompt_text

    q_marker = "Question:"
    q_idx = prompt_text.find(q_marker)
    if q_idx < 0:
        # No question marker — fall back to head-truncation (rare).
        return tokenizer.decode(ids[:max_tokens], skip_special_tokens=False)

    pre = prompt_text[:q_idx]
    post = prompt_text[q_idx:]

    post_ids = tokenizer.encode(post, add_special_tokens=False)
    # Reserve at least the entire post (question+choices+suffix).
    budget_for_pre = max_tokens - len(post_ids)
    if budget_for_pre <= 0:
        # Question + choices alone exceed budget — keep them and a tiny
        # header. Cap post itself only as a last resort.
        return tokenizer.decode(post_ids[:max_tokens], skip_special_tokens=False)

    pre_ids = tokenizer.encode(pre, add_special_tokens=False)
    if len(pre_ids) <= budget_for_pre:
        return prompt_text  # already fits, shouldn't happen

    # Keep the head of pre (intro + metadata + start of hint/lecture); drop
    # from its tail. This preserves topic/grade signal and most of the hint.
    truncated_pre = tokenizer.decode(
        pre_ids[:budget_for_pre], skip_special_tokens=False)
    return truncated_pre + post


# ── Inspect projection modules ───────────────────────────────────────────────
print("\n--- Model modules containing projection layers ---")
_proj_names = set()
for name, _ in model.named_modules():
    for key in ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]:
        if key in name:
            _proj_names.add(name)
if _proj_names:
    for n in sorted(_proj_names)[:15]:
        print(f"  {n}")
    print(f"  ... ({len(_proj_names)} total)")

2026-05-06 16:47:37.311326: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778086057.537653      83 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778086057.603178      83 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778086058.145432      83 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778086058.145469      83 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778086058.145472      83 computation_placer.cc:177] computation placer alr


Loading model: HuggingFaceTB/SmolVLM-500M-Instruct


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model loaded. Total params: 507,482,304

--- Model modules containing projection layers ---
  model.text_model.layers.0.mlp.down_proj
  model.text_model.layers.0.mlp.gate_proj
  model.text_model.layers.0.mlp.up_proj
  model.text_model.layers.0.self_attn.k_proj
  model.text_model.layers.0.self_attn.o_proj
  model.text_model.layers.0.self_attn.q_proj
  model.text_model.layers.0.self_attn.v_proj
  model.text_model.layers.1.mlp.down_proj
  model.text_model.layers.1.mlp.gate_proj
  model.text_model.layers.1.mlp.up_proj
  model.text_model.layers.1.self_attn.k_proj
  model.text_model.layers.1.self_attn.o_proj
  model.text_model.layers.1.self_attn.q_proj
  model.text_model.layers.1.self_attn.v_proj
  model.text_model.layers.10.mlp.down_proj
  ... (260 total)


In [12]:
# =============================================================================
#  4. LoRA / DoRA Setup
# =============================================================================
from peft import LoraConfig, get_peft_model, TaskType

TARGET_MODULES_MAP = {
    "qv_only":        ["q_proj", "v_proj"],
    "attention_only":  ["q_proj", "k_proj", "v_proj", "o_proj"],
    "attention_mlp":   ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    "qv_mlp":          ["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
}
target_modules = TARGET_MODULES_MAP.get(TARGET_MODULES_MODE, ["q_proj", "v_proj"])
print(f"\nLoRA target modules ({TARGET_MODULES_MODE}): {target_modules}")

lora_kwargs = dict(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=target_modules,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

if USE_DORA:
    try:
        lora_kwargs["use_dora"] = True
        _ = LoraConfig(**lora_kwargs)
        print("DoRA enabled.")
    except TypeError:
        print("[WARN] DoRA not supported. Falling back to standard LoRA.")
        del lora_kwargs["use_dora"]
        USE_DORA = False

lora_config = LoraConfig(**lora_kwargs)
model = get_peft_model(model, lora_config)

# ── Trainable parameter check ────────────────────────────────────────────────

def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable / total if total > 0 else 0
    print(f"Trainable params: {trainable:,} / {total:,} ({pct:.4f}%)")
    return trainable


trainable_count = count_trainable_parameters(model)

if trainable_count > MAX_TRAINABLE_PARAMS:
    print(f"[ERROR] {trainable_count:,} > {MAX_TRAINABLE_PARAMS:,}! Auto-reducing rank...")
    while trainable_count > MAX_TRAINABLE_PARAMS and LORA_R > 1:
        LORA_R = max(1, LORA_R // 2)
        LORA_ALPHA = max(1, LORA_ALPHA // 2)
        print(f"  Trying r={LORA_R}, alpha={LORA_ALPHA}...")
        model = AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME, torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True,
            resume_download=True,
        )
        if not torch.cuda.is_available():
            model.to(device)
        lora_kwargs["r"] = LORA_R
        lora_kwargs["lora_alpha"] = LORA_ALPHA
        lora_config = LoraConfig(**lora_kwargs)
        model = get_peft_model(model, lora_config)
        trainable_count = count_trainable_parameters(model)
    if trainable_count > MAX_TRAINABLE_PARAMS:
        raise ValueError(f"Cannot fit under {MAX_TRAINABLE_PARAMS:,} even with r=1")

assert trainable_count <= MAX_TRAINABLE_PARAMS

if USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.config.use_cache = False
    print("Gradient checkpointing enabled (use_reentrant=False).")

model.train()


LoRA target modules (attention_mlp): ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
DoRA enabled.
Trainable params: 5,088,256 / 512,570,560 (0.9927%)
[ERROR] 5,088,256 > 5,000,000! Auto-reducing rank...
  Trying r=4, alpha=8...


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Trainable params: 2,696,192 / 510,178,496 (0.5285%)
Gradient checkpointing enabled (use_reentrant=False).


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Idefics3ForConditionalGeneration(
      (model): Idefics3Model(
        (vision_model): Idefics3VisionTransformer(
          (embeddings): Idefics3VisionEmbeddings(
            (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
            (position_embedding): Embedding(1024, 768)
          )
          (encoder): Idefics3Encoder(
            (layers): ModuleList(
              (0-11): 12 x Idefics3EncoderLayer(
                (self_attn): Idefics3VisionAttention(
                  (k_proj): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=4, bias=False)
                    )
                    (

In [13]:
# =============================================================================
#  5. Training — answer-only loss (Fix A)
# =============================================================================
# Key change: we build a full user+assistant chat, then mask all tokens
# before the assistant answer with -100 so loss is only on the answer digit.

print(f"\n{'='*60}")
print("TRAINING CONFIG")
print(f"{'='*60}")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Grad accum:      {GRAD_ACCUM_STEPS}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Learning rate:   {LEARNING_RATE}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  DoRA:            {USE_DORA}")
print(f"  Image size:      {IMAGE_SIZE}")
print(f"  Inference mode:  {INFERENCE_MODE}")
print(f"  Debug:           {DEBUG}")
print(f"{'='*60}\n")


def build_train_chat(prompt_text, answer_idx):
    """
    Build a full chat with user prompt + assistant answer for training.
    Truncates the prompt text BEFORE wrapping so the answer suffix is preserved.
    """
    prompt_text = truncate_prompt_text(prompt_text)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": str(answer_idx)},
            ],
        },
    ]
    return processor.apply_chat_template(messages, add_generation_prompt=False)


def build_inference_chat(prompt_text):
    """
    Build a chat with user prompt only, ending with 'Assistant:' for generation.
    Truncates the prompt text BEFORE wrapping.
    """
    prompt_text = truncate_prompt_text(prompt_text)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text},
            ],
        },
    ]
    return processor.apply_chat_template(messages, add_generation_prompt=True)


# ── Verify chat format ───────────────────────────────────────────────────────
_sample_train_chat = build_train_chat("Test question?", 2)
_sample_infer_chat = build_inference_chat("Test question?")
print("Sample train chat (last 120 chars):")
print(f"  ...{_sample_train_chat[-120:]}")
print(f"\nSample inference chat (last 80 chars):")
print(f"  ...{_sample_infer_chat[-80:]}")

# Compute the number of answer tokens by comparing the text-only portion
# of training chat vs inference chat. This avoids expensive processor calls.
# The difference is: " {answer_idx}<end_of_utterance>\n"
# We measure it once per possible answer digit.
_answer_suffix_tokens = {}
for _a in range(10):  # 0-9 covers all possible answer indices
    _train_text = build_train_chat("Q?", _a)
    _infer_text = build_inference_chat("Q?")
    # The training chat has the answer suffix; the inference chat does not
    _train_ids = processor.tokenizer.encode(_train_text, add_special_tokens=False)
    _infer_ids = processor.tokenizer.encode(_infer_text, add_special_tokens=False)
    _answer_suffix_tokens[_a] = len(_train_ids) - len(_infer_ids)

print(f"\nAnswer suffix token counts: {_answer_suffix_tokens}")
# All should be the same (typically 3: space+digit, <end_of_utterance>, \n)
_typical_answer_len = _answer_suffix_tokens[0]
print(f"Typical answer suffix length: {_typical_answer_len} tokens")

# Verify on a real sample
_sample_item = train_ds[0]
_full_chat = build_train_chat(_sample_item["prompt"], _sample_item["answer"])
_full_inputs = processor(text=[_full_chat], images=[_sample_item["image"]],
                         return_tensors="pt", padding=False)
_full_len = _full_inputs["input_ids"].shape[1]
_ans_len = _answer_suffix_tokens[_sample_item["answer"]]
print(f"\nMasking verification (sample 0):")
print(f"  Full sequence tokens: {_full_len}")
print(f"  Answer suffix tokens: {_ans_len}")
print(f"  Prompt tokens (masked): {_full_len - _ans_len}")
_answer_ids = _full_inputs["input_ids"][0, _full_len - _ans_len:]
print(f"  Answer token IDs: {_answer_ids.tolist()}")
print(f"  Answer decoded: '{processor.tokenizer.decode(_answer_ids, skip_special_tokens=True)}'")
print(f"  Expected answer: {_sample_item['answer']}")
del _full_inputs


def train_collate_fn(batch):
    """
    Collate for training with answer-only loss.

    Strategy: we know the answer is always a short suffix at the end of the
    sequence (e.g., " 2<end_of_utterance>\\n"). We compute the suffix length
    from pure text tokenization (cheap), then mask everything except the
    last N tokens. This avoids calling the processor twice per example.
    """
    images = [item["image"] for item in batch]
    prompts = [item["prompt"] for item in batch]
    answers = [item["answer"] for item in batch]

    # Build full user+assistant chats
    formatted_texts = [
        build_train_chat(p, a)
        for p, a in zip(prompts, answers)
    ]

    inputs = processor(
        text=formatted_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    # Build labels: clone input_ids, then mask everything except answer suffix
    labels = inputs["input_ids"].clone()

    # Mask padding
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    # For each example, find where the non-pad content ends,
    # then the answer is the last N tokens before padding
    for i, ans in enumerate(answers):
        ans_len = _answer_suffix_tokens.get(ans, _typical_answer_len)
        # Find the end of actual content (first pad or end of sequence)
        seq = inputs["input_ids"][i]
        if pad_id is not None:
            non_pad_mask = (seq != pad_id)
            content_len = non_pad_mask.sum().item()
        else:
            content_len = seq.shape[0]
        # Mask everything before the answer suffix
        mask_end = content_len - ans_len
        if mask_end > 0:
            labels[i, :mask_end] = -100

    inputs["labels"] = labels
    return inputs


# Create training dataloader
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

# ── Verify masking on first batch ─────────────────────────────────────────────
_verify_batch = next(iter(train_loader))
_vb_ids = _verify_batch["input_ids"][0]
_vb_labels = _verify_batch["labels"][0]
_n_masked = (_vb_labels == -100).sum().item()
_n_total = _vb_labels.numel()
_n_active = _n_total - _n_masked
_active_tokens = _vb_labels[_vb_labels != -100]
print(f"\n--- Training label masking verification ---")
print(f"  Sequence length: {_n_total}")
print(f"  Masked tokens (prompt+padding): {_n_masked}")
print(f"  Active tokens (answer only):    {_n_active}")
if _n_active > 0 and _n_active <= 10:
    print(f"  Active token IDs:  {_active_tokens.tolist()}")
    print(f"  Active decoded:    '{processor.tokenizer.decode(_active_tokens)}'")
elif _n_active > 10:
    print(f"  [WARN] Too many active tokens ({_n_active}) — masking may not be working!")
if _n_masked == 0:
    print(f"  [ERROR] No tokens masked — answer-only loss is NOT working!")
print(f"  Expected answer:   '{train_ds[0]['answer']}'")
del _verify_batch

# ── Optimizer & Scheduler ────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM_STEPS
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Total optimizer steps: {total_steps}, warmup: {warmup_steps}")

# ── Answer Parsing (for generate fallback) ────────────────────────────────────

def parse_answer(text, num_choices):
    """Parse model output to extract a 0-indexed integer answer."""
    text = text.strip()
    if text.isdigit():
        val = int(text)
        if 0 <= val < num_choices:
            return val, True
    digits = re.findall(r'\d+', text)
    if digits:
        val = int(digits[0])
        if 0 <= val < num_choices:
            return val, True
    letters = re.findall(r'[A-Ja-j]', text)
    if letters:
        val = ord(letters[0].upper()) - ord('A')
        if 0 <= val < num_choices:
            return val, True
    return 0, False


# =============================================================================
#  Evaluation — log-likelihood ranking (Fix C) + generate fallback (Fix D)
# =============================================================================

@torch.no_grad()
def evaluate_loglikelihood(model, dataset, processor, batch_size=1):
    """
    Evaluate using log-likelihood ranking.
    For each example, compute NLL of each candidate answer ("0", "1", ...)
    and pick the one with the lowest loss (= highest probability).

    Always produces a valid integer — zero invalid outputs.
    Processes one example at a time with all its choices batched together.
    """
    model.eval()
    correct = 0
    total = 0
    predictions = []

    for idx in range(len(dataset)):
        item = dataset[idx]
        image = item["image"]
        prompt = item["prompt"]
        num_choices = item["num_choices"]
        gt_answer = item["answer"]

        # Build one chat per candidate answer
        candidate_texts = []
        for c in range(num_choices):
            chat = build_train_chat(prompt, c)
            candidate_texts.append(chat)

        # Stack images (same image repeated for each choice)
        candidate_images = [image] * num_choices

        # truncation already done inside build_train_chat
        inputs = processor(
            text=candidate_texts,
            images=candidate_images,
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in inputs.items()}

        # Build labels with prompt masked (same approach as training)
        labels = inputs["input_ids"].clone()
        pad_id = processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        for i in range(num_choices):
            ans_len = _answer_suffix_tokens.get(i, _typical_answer_len)
            seq = inputs["input_ids"][i]
            if pad_id is not None:
                content_len = (seq != pad_id).sum().item()
            else:
                content_len = seq.shape[0]
            mask_end = content_len - ans_len
            if mask_end > 0:
                labels[i, :mask_end] = -100

        # Forward pass to get per-token losses
        outputs = model(**inputs, labels=labels)

        # Compute per-candidate loss manually from logits
        logits = outputs.logits  # (num_choices, seq_len, vocab_size)
        losses = []
        for c in range(num_choices):
            # Shift: predict token t from logits at position t-1
            shift_logits = logits[c, :-1, :].contiguous()
            shift_labels = labels[c, 1:].contiguous()

            loss_fct = torch.nn.CrossEntropyLoss(reduction='sum')
            c_loss = loss_fct(shift_logits, shift_labels)

            # Count active tokens for this candidate
            n_active = (shift_labels != -100).sum().item()
            # Normalize by number of active tokens to avoid length bias
            avg_loss = c_loss.item() / max(n_active, 1)
            losses.append(avg_loss)

        pred = int(np.argmin(losses))
        predictions.append(pred)

        if gt_answer >= 0 and pred == gt_answer:
            correct += 1
        total += 1

        if total % 100 == 0:
            print(f"  Eval progress: {total}/{len(dataset)} "
                  f"(running acc: {correct/total:.4f})")

    accuracy = correct / total if total > 0 else 0.0
    model.train()
    # Log-likelihood ranking produces zero invalid predictions by design
    return accuracy, 0, predictions


@torch.no_grad()
def evaluate_generate(model, dataset, processor,
                      max_new_tokens=5, batch_size=4):
    """
    Evaluate using greedy generation (fallback).
    May produce invalid outputs that need parsing.
    """
    model.eval()
    correct = 0
    total = 0
    invalid_count = 0
    predictions = []

    for start_idx in range(0, len(dataset), batch_size):
        end_idx = min(start_idx + batch_size, len(dataset))
        batch_items = [dataset[i] for i in range(start_idx, end_idx)]

        images = [item["image"] for item in batch_items]
        prompts = [item["prompt"] for item in batch_items]

        formatted_texts = [
            build_inference_chat(p) for p in prompts
        ]

        inputs = processor(
            text=formatted_texts,
            images=images,
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in inputs.items()}

        generated_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        )

        input_len = inputs["input_ids"].shape[1]
        new_tokens = generated_ids[:, input_len:]
        decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)

        for i, output_text in enumerate(decoded):
            item = batch_items[i]
            pred, is_valid = parse_answer(output_text, item["num_choices"])
            if not is_valid:
                invalid_count += 1
            predictions.append(pred)
            if item["answer"] >= 0 and pred == item["answer"]:
                correct += 1
            total += 1

        if (start_idx // batch_size) % 50 == 0:
            print(f"  Eval progress: {total}/{len(dataset)}")

    accuracy = correct / total if total > 0 else 0.0
    model.train()
    return accuracy, invalid_count, predictions


def evaluate_val(model, dataset, processor, mode=INFERENCE_MODE, **kwargs):
    """Dispatch to the configured evaluation method."""
    if mode == "loglikelihood":
        return evaluate_loglikelihood(model, dataset, processor, **kwargs)
    else:
        return evaluate_generate(model, dataset, processor, **kwargs)




TRAINING CONFIG
  Epochs:          3
  Batch size:      2
  Grad accum:      4
  Effective batch: 8
  Learning rate:   0.0001
  LoRA r=4, alpha=8, dropout=0.1
  DoRA:            True
  Image size:      336
  Inference mode:  loglikelihood
  Debug:           False

Sample train chat (last 120 chars):
  ...<|im_start|>User:<image>Test question?<end_of_utterance>
Assistant: 2<end_of_utterance>


Sample inference chat (last 80 chars):
  ...<|im_start|>User:<image>Test question?<end_of_utterance>
Assistant:

Answer suffix token counts: {0: 4, 1: 4, 2: 4, 3: 4, 4: 4, 5: 4, 6: 4, 7: 4, 8: 4, 9: 4}
Typical answer suffix length: 4 tokens

Masking verification (sample 0):
  Full sequence tokens: 1651
  Answer suffix tokens: 4
  Prompt tokens (masked): 1647
  Answer token IDs: [216, 32, 49279, 198]
  Answer decoded: ' 0
'
  Expected answer: 0

--- Training label masking verification ---
  Sequence length: 1388
  Masked tokens (prompt+padding): 1384
  Active tokens (answer only):    4
  Active to

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
print("\nStarting training...")
best_val_acc = 0.0
best_epoch = -1
global_step = 0
optimizer_step = 0
train_losses = []  # for comparison report

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda") if use_amp else None

t_train_start = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_steps = 0
    t0 = time.time()

    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v
                 for k, v in batch.items()}

        if use_amp:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM_STEPS
            loss.backward()

        epoch_loss += loss.item() * GRAD_ACCUM_STEPS
        epoch_steps += 1
        global_step += 1

        if global_step % GRAD_ACCUM_STEPS == 0:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad()
            optimizer_step += 1

            if optimizer_step % 50 == 0:
                avg_loss = epoch_loss / epoch_steps
                elapsed = time.time() - t0
                print(
                    f"  Epoch {epoch+1}/{NUM_EPOCHS} | "
                    f"Step {optimizer_step} | "
                    f"Loss: {avg_loss:.4f} | "
                    f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                    f"Time: {elapsed:.0f}s"
                )

            if (EVAL_EVERY_N_STEPS > 0
                    and optimizer_step % EVAL_EVERY_N_STEPS == 0):
                print(f"\n  Mid-epoch validation at step {optimizer_step}...")
                val_acc, val_invalid, _ = evaluate_val(
                    model, val_ds, processor)
                print(f"  Val accuracy: {val_acc:.4f} | "
                      f"Invalid outputs: {val_invalid}")
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_epoch = epoch
                    model.save_pretrained("best_model")
                    print(f"  -> New best! (acc={val_acc:.4f})")
                model.train()

    avg_epoch_loss = epoch_loss / max(epoch_steps, 1)
    elapsed = time.time() - t0
    train_losses.append(avg_epoch_loss)
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} complete — "
          f"Avg loss: {avg_epoch_loss:.4f} | Time: {elapsed:.0f}s")

    print("Running end-of-epoch validation...")
    val_acc, val_invalid, val_preds = evaluate_val(model, val_ds, processor)
    print(f"Validation accuracy: {val_acc:.4f} | "
          f"Invalid outputs: {val_invalid}/{len(val_ds)}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        model.save_pretrained("best_model")
        print(f"-> New best model saved! (acc={val_acc:.4f})")

    if val_preds:
        dist = Counter(val_preds)
        print(f"Prediction distribution: {dict(sorted(dist.items()))}")

t_train_end = time.time()
train_time = t_train_end - t_train_start

print(f"\n{'='*60}")
print(f"Training complete. Best val accuracy: {best_val_acc:.4f} "
      f"(epoch {best_epoch+1})")
print(f"Total training time: {train_time:.0f}s")
print(f"{'='*60}")


Starting training...


/tmp/ipykernel_83/394727350.py:48: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  Epoch 1/3 | Step 50 | Loss: 4.3112 | LR: 8.62e-05 | Time: 1408s
  Epoch 1/3 | Step 100 | Loss: 2.2866 | LR: 9.96e-05 | Time: 2825s
  Epoch 1/3 | Step 150 | Loss: 1.5822 | LR: 9.83e-05 | Time: 4231s
  Epoch 1/3 | Step 200 | Loss: 1.2260 | LR: 9.60e-05 | Time: 5641s
  Epoch 1/3 | Step 250 | Loss: 1.0090 | LR: 9.28e-05 | Time: 7049s
  Epoch 1/3 | Step 300 | Loss: 0.8626 | LR: 8.87e-05 | Time: 8458s
  Epoch 1/3 | Step 350 | Loss: 0.7580 | LR: 8.38e-05 | Time: 9874s

Epoch 1/3 complete — Avg loss: 0.6951 | Time: 10955s
Running end-of-epoch validation...
  Eval progress: 100/1048 (running acc: 0.5200)
  Eval progress: 200/1048 (running acc: 0.6850)
  Eval progress: 300/1048 (running acc: 0.6967)
  Eval progress: 400/1048 (running acc: 0.7175)
  Eval progress: 500/1048 (running acc: 0.7420)
  Eval progress: 600/1048 (running acc: 0.7450)
  Eval progress: 700/1048 (running acc: 0.7557)
  Eval progress: 800/1048 (running acc: 0.7400)
  Eval progress: 900/1048 (running acc: 0.7522)
  Eval prog

In [ ]:
# =============================================================================
#  7. Test Inference & Submission
# =============================================================================

# Reload the BEST checkpoint (highest val acc) for test inference instead of
# whatever the last epoch produced. The training loop saves to "best_model"
# whenever val acc improves; without reloading we use the wrong weights.
best_model_dir = Path("best_model")
if best_model_dir.exists():
    print(f"\nReloading best LoRA adapter from {best_model_dir} "
          f"(best val acc: {best_val_acc:.4f}, epoch {best_epoch+1})...")
    from peft import PeftModel
    base_model = _load_with_retry(
        lambda: AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME,
            torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True,
            resume_download=True,
        ),
        what="AutoModelForVision2Seq (best-model reload)",
    )
    if not torch.cuda.is_available():
        base_model.to(device)
    model = PeftModel.from_pretrained(base_model, str(best_model_dir))
    model.eval()
    print("Best adapter loaded.")
else:
    print("\n[WARN] best_model dir not found; using last-epoch weights.")

# Default chosen_mode if it was not set in an earlier cell
if "chosen_mode" not in globals():
    chosen_mode = INFERENCE_MODE

print(f"\n{'='*60}")
print(f"Generating test predictions ({chosen_mode} mode)...")
print(f"{'='*60}")

model.eval()

if chosen_mode == "loglikelihood":
    _, test_invalid, test_preds = evaluate_loglikelihood(
        model, test_ds, processor
    )
    test_predictions = [
        {"id": test_ds[i]["id"], "answer": test_preds[i]}
        for i in range(len(test_ds))
    ]
else:
    test_predictions = []
    test_invalid = 0
    with torch.no_grad():
        for start_idx in range(0, len(test_ds), 4):
            end_idx = min(start_idx + 4, len(test_ds))
            batch_items = [test_ds[i] for i in range(start_idx, end_idx)]
            images = [item["image"] for item in batch_items]
            prompts = [item["prompt"] for item in batch_items]

            formatted_texts = [
                build_inference_chat(p) for p in prompts
            ]
            inputs = processor(
                text=formatted_texts, images=images,
                return_tensors="pt", padding=True,
            )
            inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                      for k, v in inputs.items()}

            generated_ids = model.generate(
                **inputs, max_new_tokens=5, do_sample=False,
            )
            input_len = inputs["input_ids"].shape[1]
            new_tokens = generated_ids[:, input_len:]
            decoded = processor.batch_decode(
                new_tokens, skip_special_tokens=True)

            for i, output_text in enumerate(decoded):
                item = batch_items[i]
                pred, is_valid = parse_answer(
                    output_text, item["num_choices"])
                if not is_valid:
                    test_invalid += 1
                test_predictions.append(
                    {"id": item["id"], "answer": pred})

            if (start_idx // 4) % 100 == 0:
                print(f"  Test progress: "
                      f"{len(test_predictions)}/{len(test_ds)}")

print(f"Test inference complete. Invalid: {test_invalid}/{len(test_ds)}")

In [ ]:
# ── Build & validate submission ──────────────────────────────────────────────
submission = pd.DataFrame(test_predictions)

# In debug mode with TEST_SUBSET, we only predicted a subset
if DEBUG and TEST_SUBSET is not None:
    _full_test_df = pd.read_csv(DATA_DIR / "test.csv")
    _full_test_df["choices"] = _full_test_df["choices"].apply(json.loads)
    _remaining = _full_test_df.iloc[len(test_predictions):]
    _fallback = pd.DataFrame({
        "id": _remaining["id"],
        "answer": 0,
    })
    submission = pd.concat([submission, _fallback], ignore_index=True)
    _full_test = _full_test_df
else:
    _full_test = test_df if not DEBUG else pd.read_csv(DATA_DIR / "test.csv")
    if isinstance(_full_test["choices"].iloc[0], str):
        _full_test["choices"] = _full_test["choices"].apply(json.loads)

# For validation, always compare against the full test set
_full_test_for_validation = pd.read_csv(DATA_DIR / "test.csv")
_full_test_for_validation["choices"] = _full_test_for_validation["choices"].apply(json.loads)

# Ensure submission covers all test IDs
if len(submission) < len(_full_test_for_validation):
    _existing_ids = set(submission["id"])
    _missing = _full_test_for_validation[
        ~_full_test_for_validation["id"].isin(_existing_ids)
    ]
    _fallback = pd.DataFrame({"id": _missing["id"], "answer": 0})
    submission = pd.concat([submission, _fallback], ignore_index=True)

# Reorder to match test.csv ordering
submission = submission.set_index("id").loc[
    _full_test_for_validation["id"]
].reset_index()

assert list(submission.columns) == ["id", "answer"], \
    f"Columns: {list(submission.columns)}"
assert len(submission) == len(_full_test_for_validation), \
    f"Length mismatch: {len(submission)} vs {len(_full_test_for_validation)}"
assert submission["id"].tolist() == _full_test_for_validation["id"].tolist(), \
    "ID mismatch"
assert submission["answer"].notna().all(), "Missing answers"

for idx, (pred, n) in enumerate(
    zip(submission["answer"], _full_test_for_validation["num_choices"])
):
    assert 0 <= int(pred) < int(n), \
        f"Row {idx}: answer {pred} out of [0, {n})"

submission["answer"] = submission["answer"].astype(int)
submission.to_csv("submission.csv", index=False)
print(f"\nSaved submission.csv ({len(submission)} rows)")
print(submission.head(10))

test_dist = Counter(submission["answer"].tolist())
print(f"\nTest prediction distribution: {dict(sorted(test_dist.items()))}")


In [ ]:
# =============================================================================
#  8. Final Summary / Comparison Report
# =============================================================================
print(f"\n{'='*60}")
print("FINAL SUMMARY — REVISED PIPELINE")
print(f"{'='*60}")
trainable_final = count_trainable_parameters(model)
print(f"Training losses by epoch:     {[f'{l:.4f}' for l in train_losses]}")
print(f"Val accuracy (loglikelihood):  {final_val_acc:.4f}")
print(f"Val accuracy (generate):       {gen_val_acc:.4f}")
print(f"Val invalid (loglikelihood):   {final_val_invalid}")
print(f"Val invalid (generate):        {gen_val_invalid}")
print(f"Test invalid ({chosen_mode}):  {test_invalid}")
print(f"Total training time:           {train_time:.0f}s")
print(f"Inference mode used:           {chosen_mode}")
print(f"submission.csv passes checks:  True")
print(f"{'='*60}")

print(f"""
COMPARISON vs BASELINE (starter_notebook_baseline.py):
------------------------------------------------------
Metric                   | Baseline             | Revised
-------------------------|----------------------|---------------------
Trainable params         | (same LoRA config)   | {trainable_final:,}
Training objective       | Full-sequence loss   | Answer-only loss
Label masking            | Pad tokens only      | Prompt + pad masked
Inference method         | Generate + parse     | Log-likelihood rank
Invalid predictions      | Varies (parse-based) | 0 (by design)
Val accuracy (loglik)    | N/A                  | {final_val_acc:.4f}
Val accuracy (generate)  | (run baseline)       | {gen_val_acc:.4f}
Test invalid             | (run baseline)       | {test_invalid}
submission.csv valid     | (run baseline)       | True
------------------------------------------------------

To compare fairly, run both scripts with the same config:
  DEBUG=True, TRAIN_SUBSET=64, VAL_SUBSET=64, same LoRA, same seed.

Recommendation:
  - If revised val accuracy >= baseline: keep revised.
  - If revised val accuracy ~= baseline but invalid drops: keep revised.
  - Log-likelihood ranking eliminates invalid outputs entirely.
""")